## Corrections log (review notes)

| Cell | Problem found in the original | Fix |
|---|---|---|
| LightGBM | A block named "FORCE ONE ERROR IF ACCURACY IS 100%" flipped a prediction, so the printed 99.17 % test accuracy and the report were **artificial**. | Removed. Metrics are now computed on the model's real predictions. |
| Cleaning / split | The `id` column was used as a feature. In this dataset `id` is ordered by class, so it leaks the label (it is the top feature in both the SHAP and LIME plots). | `id` is dropped. |
| Cleaning / split | Imputation, encoding and scaling were fitted on the full data **before** the split (test-set leakage). | Split first; all statistics are learned from the training set only. |
| Labels | `LabelEncoder` gave `ckd`=0 and `notckd`=1, yet the risk table treated column 1 as "disease probability" and LIME named class 0 "Normal". Both were inverted. Stray whitespace (e.g. `ckd\t`) also created extra classes that were merged by `replace(2, 1)`. | Text is stripped and the target is mapped explicitly: `notckd`=0, `ckd`=1. The `replace(2, 1)` workarounds are removed. |
| SHAP | The whole block was duplicated. | De-duplicated. |
| LIME | Title and class names said "DKD"; the dataset label is CKD. | Renamed to CKD / Not CKD. |
| Pickle | Only the model was saved, so it cannot be reused without the preprocessing. | Preprocessing objects are saved too. |

**All outputs below the data-loading cell were cleared on purpose.** They came from the old, flawed code, and no new results are shown until the notebook is re-run on your `dataset.csv`.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report
import lightgbm as lgb
import matplotlib.pyplot as plt

# Colab-specific import for file upload
from google.colab import files
import io

# Upload and read dataset
uploaded = files.upload()
if not uploaded:
    raise Exception("No file selected. Please upload a dataset.")
for fn in uploaded.keys():
    df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))
print("Dataset loaded successfully!")
print("Shape of data:", df.shape)
print(df.head())

Saving dataset.csv to dataset.csv
Dataset loaded successfully!
Shape of data: (400, 26)
   id   age    bp     sg   al   su     rbc        pc         pcc          ba  \
0   0  48.0  80.0  1.020  1.0  0.0     NaN    normal  notpresent  notpresent   
1   1   7.0  50.0  1.020  4.0  0.0     NaN    normal  notpresent  notpresent   
2   2  62.0  80.0  1.010  2.0  3.0  normal    normal  notpresent  notpresent   
3   3  48.0  70.0  1.005  4.0  0.0  normal  abnormal     present  notpresent   
4   4  51.0  80.0  1.010  2.0  0.0  normal    normal  notpresent  notpresent   

   ...  pcv    wc   rc  htn   dm  cad appet   pe  ane classification  
0  ...   44  7800  5.2  yes  yes   no  good   no   no            ckd  
1  ...   38  6000  NaN   no   no   no  good   no   no            ckd  
2  ...   31  7500  NaN   no  yes   no  poor   no  yes            ckd  
3  ...   32  6700  3.9  yes   no   no  poor  yes  yes            ckd  
4  ...   35  7300  4.6   no   no   no  good   no   no            ckd  

[5 r

In [ ]:
# Stateless cleaning only (nothing is learned from the data here)
df = df.copy()

# 'id' is just a row index; in this dataset it is ordered by class, so it leaks the label
df = df.drop(columns=['id'], errors='ignore')

# Strip stray whitespace/tabs (e.g. 'ckd\t', '\tno') and treat '?' / empty strings as missing
for col in df.select_dtypes(exclude='number').columns:
    df[col] = (df[col].astype(str).str.strip()
                 .replace({'?': np.nan, '': np.nan, 'nan': np.nan}))
df = df.dropna(how='all')

# Target: explicit mapping so that 1 = disease (CKD), 0 = no disease
target_col = 'classification'
df[target_col] = df[target_col].map({'notckd': 0, 'ckd': 1})
assert df[target_col].notna().all(), "Unexpected values in the target column"

# Columns stored as text but holding numbers (e.g. pcv, wc, rc) -> numeric
for col in df.columns.drop(target_col):
    converted = pd.to_numeric(df[col], errors='coerce')
    if converted.notna().sum() == df[col].notna().sum():   # every non-missing value is numeric
        df[col] = converted

print("\n Data cleaned (no statistics learned yet)")
print("Class balance (0 = notckd, 1 = ckd):\n", df[target_col].value_counts())

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

# Split FIRST, then learn imputation / encoding / scaling from the training set only
X = df.drop(columns=[target_col])
y = df[target_col].astype(int)

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

cat_cols = X.select_dtypes(exclude='number').columns.tolist()
num_cols = [c for c in X.columns if c not in cat_cols]

# Imputation values from the training set
num_medians = X_train_raw[num_cols].median()
cat_modes = X_train_raw[cat_cols].mode().iloc[0]

def impute(d):
    d = d.copy()
    d[num_cols] = d[num_cols].fillna(num_medians)
    d[cat_cols] = d[cat_cols].fillna(cat_modes)
    return d

X_train_imp, X_test_imp = impute(X_train_raw), impute(X_test_raw)

# Encode categorical columns (unseen categories in the test set -> -1)
encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
X_train_enc, X_test_enc = X_train_imp.copy(), X_test_imp.copy()
if cat_cols:
    X_train_enc[cat_cols] = encoder.fit_transform(X_train_imp[cat_cols])
    X_test_enc[cat_cols] = encoder.transform(X_test_imp[cat_cols])

# Normalize features (scaler fitted on the training set only)
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train_enc)
X_test = scaler.transform(X_test_enc)

print("\n Data split into Train and Test sets")
print("Train:", X_train.shape, " Test:", X_test.shape)
print("Train class counts:", dict(y_train.value_counts()), " Test class counts:", dict(y_test.value_counts()))

In [ ]:
from sklearn.metrics import classification_report, accuracy_score
import lightgbm as lgb

# Train model
model_lgb = lgb.LGBMClassifier(
    objective='binary',
    random_state=42,
    verbosity=-1
)
model_lgb.fit(X_train, y_train)

# Predict (real predictions, no manual changes)
y_pred_lgb = model_lgb.predict(X_test)

train_acc = model_lgb.score(X_train, y_train) * 100
test_acc = accuracy_score(y_test, y_pred_lgb) * 100

print("\nLightGBM Results:")
print("Training Accuracy:", round(train_acc, 2), "%")
print("Test Accuracy:", round(test_acc, 2), "%")
print("Classification Report:\n",
      classification_report(y_test, y_pred_lgb, target_names=['Not CKD', 'CKD'], zero_division=0))

In [ ]:
# Probability of the disease class (class 1 = CKD)
risk_prob = model_lgb.predict_proba(X_test)[:, 1]

# Risk Stratification Function
def risk_level(prob):
    if prob < 0.33:
        return "Low Risk"
    elif prob < 0.66:
        return "Medium Risk"
    else:
        return "High Risk"

risk_labels = [risk_level(p) for p in risk_prob]

risk_results = pd.DataFrame({
    "Actual Label (1 = CKD)": y_test.values,
    "Predicted Probability": risk_prob,
    "Risk Level": risk_labels
})

print("\nRisk Stratification Results:")
print(risk_results.head())
print("\nRisk level vs actual label:")
print(pd.crosstab(risk_results["Risk Level"], risk_results["Actual Label (1 = CKD)"]))

In [ ]:
import shap

feature_names = list(X.columns)
X_train_df = pd.DataFrame(X_train, columns=feature_names)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

explainer = shap.Explainer(model_lgb, X_train_df)
shap_values = explainer(X_test_df)

# Some shap versions return one slice per class for binary models; keep the CKD class (1)
if shap_values.values.ndim == 3:
    shap_values = shap_values[:, :, 1]

# GLOBAL EXPLANATION
shap.plots.beeswarm(shap_values)

# LOCAL EXPLANATION (first test patient)
shap.initjs()
shap.force_plot(
    shap_values.base_values[0],
    shap_values.values[0],
    X_test_df.iloc[0]
)

In [7]:
!pip install lime

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283913 sha256=21f4f5b3a2f4fc3ce1669a6c675f6e9af6b28374780365e707936dc9be147334
  Stored in directory: /root/.cache/pip/wheels/7c/04/5c/157dc9106512a6c7a30653ec064490c94a49e0fc8f63d19ab9
Successfully built lime


In [ ]:
from lime.lime_tabular import LimeTabularExplainer
import matplotlib.pyplot as plt

lime_explainer = LimeTabularExplainer(
    training_data=X_train,
    feature_names=list(X.columns),
    class_names=['Not CKD', 'CKD'],
    mode='classification'
)

# Explain one test sample
i = 0  # patient index
exp = lime_explainer.explain_instance(X_test[i], model_lgb.predict_proba)

fig = exp.as_pyplot_figure()
plt.title("LIME Explanation for CKD Prediction")
plt.tight_layout()
plt.show()

In [ ]:
import pickle

# Save the model AND the preprocessing needed to reproduce its inputs
with open('model_lgb.pkl', 'wb') as file:
    pickle.dump(model_lgb, file)

preprocessing = {
    "feature_columns": list(X.columns),
    "num_cols": num_cols, "cat_cols": cat_cols,
    "num_medians": num_medians, "cat_modes": cat_modes,
    "encoder": encoder, "scaler": scaler,
    "label_map": {"notckd": 0, "ckd": 1},
}
with open('preprocessing.pkl', 'wb') as file:
    pickle.dump(preprocessing, file)

print("Model saved to model_lgb.pkl and preprocessing saved to preprocessing.pkl")